<a href="https://colab.research.google.com/github/Rheaxu/story-converter-ai/blob/main/K5ReadingStoryConverter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1: Install Dependencies
Install PyMuPDF for PDF processing and edge-tts for high-quality neural voice generation.

In [ ]:
### Step 1: Install Dependencies
# Install PyMuPDF for PDF processing and edge-tts for high-quality voice generation.
!pip install pymupdf edge-tts

# Step 2a: Public Story Converter Core Logic
This tool extracts text and images from your PDFs and generates a standard interactive HTML website for public hosting.

In [ ]:
### Step 2a: Public Story Converter Core Logic
# Final fix for transparency and overlapping text artifacts.

import fitz
import os
import base64
import json
import asyncio
import ipywidgets as widgets
from IPython.display import display
import edge_tts

def extract_pdf_data(pdf_path):
    doc = fitz.open(pdf_path)
    story_data = []
    for page in doc:
        page_rect = page.rect
        page_content = {
            "text_blocks": [],
            "word_bank": [],
            "images": [],
            "dimensions": [page_rect.width, page_rect.height]
        }

        # Use get_image_info to find exact locations
        img_infos = page.get_image_info(hashes=False, xrefs=True)
        for info in img_infos:
            bbox = info['bbox']
            try:
                # FIX: Render image area with alpha AND hide text/annots to prevent 'burned-in' text
                pix = page.get_pixmap(
                    clip=bbox,
                    alpha=True,
                    matrix=fitz.Matrix(2, 2),
                    annots=False
                )

                img_data = pix.tobytes("png")
                img_base64 = base64.b64encode(img_data).decode('utf-8')

                page_content["images"].append({
                    "base64": img_base64,
                    "bbox": bbox
                })
                pix = None
            except Exception as e:
                print(f"Skipping image at {bbox} due to: {e}")

        # Now extract text normally
        blocks = page.get_text("dict")["blocks"]
        for b in blocks:
            if b["type"] == 0: # text type
                for l in b["lines"]:
                    for s in l["spans"]:
                        clean_text = s["text"].strip()
                        if not clean_text: continue
                        x0, y0, x1, y1 = s["bbox"]

                        # UPDATED LOGIC: Target top 25% height and right 70% width for word bank
                        is_in_potential_word_bank_area = (y0 < page_rect.height * 0.25) and (x0 > page_rect.width * 0.30)
                        # Allow short phrases (up to 5 words) to be split into individual word bank items
                        is_short_text = (len(clean_text.split()) <= 5)

                        is_footer = (y0 < page_rect.height * 0.08) or (y0 > page_rect.height * 0.92)
                        is_reader_label = any(x in clean_text.lower() for x in ["level", "reader"])

                        data = {
                            "text": s["text"],
                            "bbox": s["bbox"],
                            "size": s["size"],
                            "font": s["font"],
                            "is_footer": is_footer or is_reader_label
                        }

                        # A text span is considered a word bank item if it's in the potential area,
                        # is short text, and is not a footer or reader label.
                        if is_in_potential_word_bank_area and is_short_text and not is_footer and not is_reader_label:
                            words_in_span = clean_text.split() # Split into individual words
                            for word in words_in_span:
                                clean_w = word.strip(",.?!:;")
                                if clean_w:
                                    # Add to word_bank only if it's not already there to prevent duplicates
                                    if clean_w not in page_content["word_bank"]:
                                        page_content["word_bank"].append(clean_w)
                        else:
                            page_content["text_blocks"].append(data)
        story_data.append(page_content)
    return story_data

# Helper for generating audio
async def generate_audio(text, output_path, voice="en-US-GuyNeural"):
    communicate = edge_tts.Communicate(text, voice, rate="-20%")
    await communicate.save(output_path)

def generate_html(pdf_name, story_data, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    audio_dir_name = f"audio_{pdf_name.replace('.pdf', '')}"
    audio_dir_path = os.path.join(output_folder, audio_dir_name)
    os.makedirs(audio_dir_path, exist_ok=True)

    html_template = """<html><head><meta charset=\"utf-8\"><style>
        body { background: #f0f2f5; display: flex; flex-direction: column; align-items: center; padding: 20px; margin: 0; font-family: 'Segoe UI', sans-serif; }
        .word-bank { background: #fff; width: 90%; max-width: 950px; padding: 20px; margin: 20px 0; border-radius: 12px; box-shadow: 0 4px 10px rgba(0,0,0,0.1); display: flex; flex-wrap: wrap; gap: 12px; justify-content: center; border: 4px solid #4a90e2; }
        .word-bank h2 { width: 100%; text-align: center; color: #4a90e2; font-size: 40px; font-weight: 800; }
        .word-item { padding: 10px 20px; background: #f8f9fa; color: #4a90e2; border-radius: 8px; font-weight: bold; cursor: pointer; border: 2px solid #4a90e2; font-size: 2rem; }
        .page { background: white; position: relative; margin-bottom: 40px; box-shadow: 0 5px 15px rgba(0,0,0,0.2); overflow: hidden; }
        .absolute-element { position: absolute; white-space: nowrap; transform-origin: 0 0; line-height: 1; }
        .sentence { cursor: pointer; z-index: 10; font-weight: 500; font-size: 50px !important; border-radius: 8px; }
        .story-title { font-size: 70px !important; font-weight: 700 !important; }
        .footer-text { z-index: 5; opacity: 0.6; pointer-events: none; }
        .playing { background: rgba(74, 144, 226, 0.3) !important; }
        img { z-index: 1; }
    </style><script>function playAudio(el, f){ const audio = new Audio(f); el.classList.add('playing'); audio.onended = () => el.classList.remove('playing'); audio.play(); }</script></head><body>"""

    sentence_count = 0
    loop = asyncio.get_event_loop()
    unique_words = []
    vocab_html = '<div class="word-bank"><h2>Word Bank</h2>'
    for p in story_data:
        for txt in p["word_bank"]:
            if txt not in unique_words:
                unique_words.append(txt)
                sentence_count += 1
                audio_file = f"vocab_{sentence_count}.mp3"
                audio_path = os.path.join(audio_dir_path, audio_file)
                if loop.is_running(): loop.create_task(generate_audio(txt, audio_path))
                else: asyncio.run(generate_audio(txt, audio_path))
                vocab_html += f'<div class="word-item" onclick="playAudio(this, \'{audio_dir_name}/{audio_file}\')">{txt}</div>'

    if unique_words: html_template += vocab_html + '</div>'

    is_first_sentence = True
    for p in story_data:
        w, h = p["dimensions"]
        scale = 2.0
        html_template += f'<div class="page" style="width: {w * scale}px; height: {h * scale}px;">'
        for img in p["images"]:
            ix0, iy0, ix1, iy1 = img["bbox"]
            html_template += f'<img src="data:image/png;base64,{img["base64"]}" style="position:absolute; left:{ix0 * scale}px; top:{iy0 * scale}px; width:{(ix1-ix0) * scale}px; height:{(iy1-iy0) * scale}px;">'

        for txt in p["text_blocks"]:
            x0, y0, x1, y1 = txt["bbox"]
            if txt["is_footer"]:
                style = f"left:{x0 * scale}px; top:{y0 * scale}px; font-size:{txt['size'] * scale * 0.5}px;"
                html_template += f'<div class="absolute-element footer-text" style="{style}">{txt["text"]}</div>'
            else:
                style = f"left:{x0 * scale}px; top:{y0 * scale}px;"
                sentence_count += 1
                audio_file = f"s_{sentence_count}.mp3"
                audio_path = os.path.join(audio_dir_path, audio_file)
                if loop.is_running(): loop.create_task(generate_audio(txt["text"].strip(), audio_path))
                else: asyncio.run(generate_audio(txt["text"].strip(), audio_path))
                cls = "sentence story-title" if is_first_sentence else "sentence"
                html_template += f'<div class="absolute-element {cls}" style="{style}" onclick="playAudio(this, \'{audio_dir_name}/{audio_file}\')">{txt["text"]}</div>'
                is_first_sentence = False
        html_template += '</div>'

    html_template += "</body></html>"
    return html_template

pdf_list = [f for f in os.listdir(".") if f.endswith(".pdf")]
dropdown = widgets.Dropdown(options=pdf_list if pdf_list else ["No PDFs found"], description="Select PDF:")
folder_input = widgets.Text(value="final_output", description="Output Folder:")
button = widgets.Button(description="Convert", button_style="primary")
output = widgets.Output()

def on_click(b):
    with output:
        output.clear_output()
        if dropdown.value == "No PDFs found": return
        print(f"Processing {dropdown.value} (Clean Layer Fix Applied)... ")
        try:
            data = extract_pdf_data(dropdown.value)
            html = generate_html(dropdown.value, data, folder_input.value)
            output_path = os.path.join(folder_input.value, dropdown.value.replace(".pdf", ".html"))
            with open(output_path, "w", encoding='utf-8') as f:
                f.write(html)
            print(f"Success! Interactive story generated at: {output_path}")
        except Exception as e:
            print(f"Error: {e}")

button.on_click(on_click)
display(dropdown, folder_input, button, output)

# Step 2b: Private Story Converter Core Logic
This version allows you to include a **SAS Token**. The token is appended to audio URLs in the HTML, enabling the interactive features to work even when assets are hosted in private cloud storage.

In [ ]:
### Step 2b: Private Story Converter (Dynamic URL & SAS Integration)

import fitz, os, base64, asyncio, edge_tts
import ipywidgets as widgets
from IPython.display import display

async def generate_private_story_async(pdf_name, story_data, output_folder):
    story_id = pdf_name.replace('.pdf', '')
    story_dir = os.path.join(output_folder, story_id)
    os.makedirs(story_dir, exist_ok=True)
    audio_dir_name = f"audio_{story_id}"
    audio_dir_path = os.path.join(story_dir, audio_dir_name)
    os.makedirs(audio_dir_path, exist_ok=True)

    # CSS braces are doubled for f-string compatibility
    html_template = f"""<html><head><meta charset="utf-8"><style>
        body {{ background: #f0f2f5; display: flex; flex-direction: column; align-items: center; padding: 20px; margin: 0; font-family: 'Segoe UI', sans-serif; }}
        .word-bank {{ background: #fff; width: 90%; max-width: 950px; padding: 20px; margin: 20px 0; border-radius: 12px; box-shadow: 0 4px 10px rgba(0,0,0,0.1); display: flex; flex-wrap: wrap; gap: 12px; justify-content: center; border: 4px solid #4a90e2; }}
        .word-bank h2 {{ width: 100%; text-align: center; color: #4a90e2; font-size: 40px; font-weight: 800; }}
        .word-item {{ padding: 10px 20px; background: #f8f9fa; color: #4a90e2; border-radius: 8px; font-weight: bold; cursor: pointer; border: 2px solid #4a90e2; font-size: 2rem; }}
        .page {{ background: white; position: relative; margin-bottom: 40px; box-shadow: 0 5px 15px rgba(0,0,0,0.2); overflow: hidden; }}
        .absolute-element {{ position: absolute; white-space: nowrap; transform-origin: 0 0; line-height: 1; }}
        .sentence {{ cursor: pointer; z-index: 10; font-weight: 500; font-size: 50px !important; border-radius: 8px; }}
        .story-title {{ font-size: 70px !important; font-weight: 700 !important; }}
        .footer-text {{ z-index: 5; opacity: 0.6; pointer-events: none; }}
        .playing {{ background: rgba(74, 144, 226, 0.3) !important; !important; }}
        img {{ z-index: 1; }}
    </style><script>function playAudio(el, filename){{ const currentUrl = window.location.href.split('?')[0]; const baseUrl = currentUrl.substring(0, currentUrl.lastIndexOf('/') + 1); const sasToken = window.location.search; const audioUrl = baseUrl + "{audio_dir_name}/" + filename + sasToken; const audio = new Audio(audioUrl); el.classList.add('playing'); audio.onended = () => el.classList.remove('playing'); audio.play(); }}</script></head><body>"""

    audio_tasks, unique_words, sentence_count = [], [], 0
    vocab_html = '<div class="word-bank"><h2>Word Bank</h2>'
    for p in story_data:
        for txt in p["word_bank"]:
            if txt not in unique_words:
                unique_words.append(txt)
                sentence_count += 1
                audio_file = f"vocab_{sentence_count}.mp3"
                audio_tasks.append(generate_audio(txt, os.path.join(audio_dir_path, audio_file)))
                vocab_html += f'<div class="word-item" onclick="playAudio(this, \'{audio_file}\')">{txt}</div>'

    if unique_words: html_template += vocab_html + '</div>'

    is_first_sentence = True
    for p in story_data:
        w, h = p["dimensions"]
        scale = 2.0
        html_template += f'<div class="page" style="width: {w * scale}px; height: {h * scale}px;">'
        for img in p["images"]:
            ix0, iy0, ix1, iy1 = img["bbox"]
            html_template += f'<img src="data:image/png;base64,{img["base64"]}" style="position:absolute; left:{ix0 * scale}px; top:{iy0 * scale}px; width:{(ix1-ix0) * scale}px; height:{(iy1-iy0) * scale}px;">'
        for txt in p["text_blocks"]:
            x0, y0, x1, y1 = txt["bbox"]
            if txt["is_footer"]:
                # Fixed syntax error: removed backslashes from inside the f-string dictionary access
                style = f"left:{x0 * scale}px; top:{y0 * scale}px; font-size:{txt['size'] * scale * 0.5}px;"
                html_template += f'<div class="absolute-element footer-text" style="{style}">{txt["text"]}</div>'
            else:
                style = f"left:{x0 * scale}px; top:{y0 * scale}px;"
                sentence_count += 1
                audio_file = f"s_{sentence_count}.mp3"
                audio_tasks.append(generate_audio(txt["text"].strip(), os.path.join(audio_dir_path, audio_file)))
                cls = "sentence story-title" if is_first_sentence else "sentence"
                html_template += f'<div class="absolute-element {cls}" style="{style}" onclick="playAudio(this, \'{audio_file}\')">{txt["text"]}</div>'
                is_first_sentence = False
        html_template += '</div>'

    html_template += "</body></html>"

    # Wait for all audio generation tasks to complete before writing the final HTML
    if audio_tasks:
        await asyncio.gather(*audio_tasks)

    with open(os.path.join(story_dir, "index.html"), "w", encoding='utf-8') as f:
        f.write(html_template)

    return os.path.join(story_dir, "index.html")

dropdown_pa = widgets.Dropdown(options=[f for f in os.listdir(".") if f.endswith(".pdf")], description="Select PDF:")
button_pa = widgets.Button(description="Generate Private Story", button_style="warning")
output_pa = widgets.Output()

def on_click_pa(b):
    with output_pa:
        output_pa.clear_output()
        if not dropdown_pa.value: return
        print(f"Starting conversion for {dropdown_pa.value}...")

        async def run_process():
            try:
                # extract_pdf_data is defined in Step 2a
                data = extract_pdf_data(dropdown_pa.value)
                result_path = await generate_private_story_async(dropdown_pa.value, data, "private_build")
                print(f"Success! Private website generated at: {result_path}")
            except Exception as e:
                print(f"Error: {e}")

        asyncio.ensure_future(run_process())

button_pa.on_click(on_click_pa)
display(dropdown_pa, button_pa, output_pa)

# Step 3a: Package for Public Download
Gather all your standard HTML and audio assets into a single ZIP file.

In [ ]:
### Step 3: Package for Download
# This cell gathers your HTML and audio assets into a single ZIP file for easy downloading.

import zipfile
import os
from google.colab import files as colab_files

def package_complete_story(output_filename="story_web_package.zip"):
    with zipfile.ZipFile(output_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        found_files = 0
        for root, dirs, files in os.walk('.'):
            if '.config' in root or 'sample_data' in root: continue
            for file in files:
                if file.endswith('.html') or file.endswith('.mp3'):
                    file_path = os.path.join(root, file)
                    arcname = os.path.relpath(file_path, '.')
                    zipf.write(file_path, arcname)
                    found_files += 1

    if found_files > 0:
        print(f"Packaged {found_files} assets into {output_filename}")
        colab_files.download(output_filename)
    else:
        print("No HTML or MP3 files found to package. Please run the conversion first.")

package_btn = widgets.Button(description='Download Package (ZIP)', button_style='success')
package_out = widgets.Output()

def on_pkg_click(b):
    with package_out:
        package_out.clear_output()
        package_complete_story()

package_btn.on_click(on_pkg_click)
display(package_btn, package_out)

# Step 3b: Page for Private Download
Package the HTML and assets generated by the Private Story Converter.

In [ ]:
### Step 3b: Package Private Story (ZIP)

package_btn_p = widgets.Button(description='Download Private Package', button_style='success')
package_out_p = widgets.Output()

def on_pkg_p_click(b):
    with package_out_p:
        package_out_p.clear_output()
        package_complete_story("private_story_package.zip")

package_btn_p.on_click(on_pkg_p_click)
display(package_btn_p, package_out_p)

# Step 4: Asset Cleanup Tool
Use this tool to delete generated folders if you want to start fresh or save disk space.

In [ ]:
### Step 4: Asset Cleanup Tool
# Use this cell to delete generated folders if you want to start fresh or save space.

import shutil

def delete_directory(directory_path):
    if os.path.exists(directory_path) and os.path.isdir(directory_path):
        try:
            shutil.rmtree(directory_path)
            print(f"Successfully deleted: {directory_path}")
        except Exception as e:
            print(f"Error deleting {directory_path}: {e}")
    else:
        print(f"Directory not found: {directory_path}")

delete_target = widgets.Text(placeholder='folder_name', description='Delete Folder:')
delete_btn = widgets.Button(description='Delete Folder', button_style='danger')
delete_out = widgets.Output()

def on_delete_click(b):
    with delete_out:
        delete_out.clear_output()
        if delete_target.value:
            delete_directory(delete_target.value)
        else:
            print("Please enter a folder name (e.g., audio_ka1_CatAndRat)")

delete_btn.on_click(on_delete_click)
display(delete_target, delete_btn, delete_out)